# Embeddings & Vector Spaces

## Overview

This document explains how the `embed_texts()` function serves as the bridge between human-readable text and the vector space where similarity search happens. This function takes a list of strings, sends them to Ollama's embedding model, and gets back numerical vectors that capture semantic meaning.

This is the key insight behind RAG: you can compare the *meaning* of a user's question to the *meaning* of document chunks by comparing their vector representations. Two texts about the same topic will have similar vectors, even if they use completely different words.

## Architecture Context

Both services use embeddings:
- **Ingestion API** embeds each text chunk before storing in Qdrant (`embedder.py`)
- **Chat API** embeds the user's question before searching Qdrant (`chain.py`)

The embedding model (nomic-embed-text) produces 768-dimensional vectors. Both services must use the same model — if you embed documents with one model and search with another, the vectors won't be comparable.

In [ ]:
# Prerequisites — requires Ollama running with nomic-embed-text
import httpx
import asyncio
import numpy as np

# Check Ollama connectivity
OLLAMA_BASE_URL = "http://localhost:11434"

async def check_ollama():
    try:
        async with httpx.AsyncClient() as client:
            response = await client.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5.0)
            models = [m["name"] for m in response.json().get("models", [])]
            if any("nomic-embed-text" in m for m in models):
                print(f"✓ Ollama is running with nomic-embed-text")
            else:
                print(f"✗ nomic-embed-text not found. Run: ollama pull nomic-embed-text")
                print(f"  Available models: {models}")
    except Exception as e:
        print(f"✗ Cannot reach Ollama at {OLLAMA_BASE_URL}: {e}")
        print(f"  Make sure Ollama is running (ollama serve)")

await check_ollama()

## Package Introductions

### httpx
httpx is Python's modern HTTP client — think of it as Go's `http.Client` or Node's `fetch`/`axios`. We chose it over the older `requests` library because:
- **Async support** — `httpx.AsyncClient()` works with `async/await`, which we need inside FastAPI handlers. The `requests` library is sync-only.
- **Streaming support** — `client.stream()` is used in the Streaming & SSE document for SSE. httpx handles this natively.
- **Similar API to requests** — if you've seen Python projects using `requests`, httpx is almost identical but adds async.

Key APIs:
- `httpx.AsyncClient()` — create an async client (use as context manager: `async with httpx.AsyncClient() as client:`)
- `client.post(url, json={...})` — send a POST with JSON body
- `response.json()` — parse JSON response
- `response.raise_for_status()` — raise exception on 4xx/5xx (like checking `resp.StatusCode` in Go)

### numpy
numpy is Python's numerical computing library. We only use it here for one thing: computing cosine similarity between vectors. It's the standard way to do vector math in Python.

Key API: `np.dot()` for dot product, `np.linalg.norm()` for vector magnitude.

## Go/TS Comparison

| Concept | Go | Python |
|---------|-----|--------|
| HTTP client | `http.Client{}` | `httpx.AsyncClient()` |
| POST with JSON | `json.Marshal` + `http.NewRequest` | `client.post(url, json=data)` |
| Context manager | `defer resp.Body.Close()` | `async with client:` (auto-closes) |
| Async I/O | goroutines (implicit) | `async/await` (explicit) |

The biggest difference: In Go, every HTTP call is implicitly concurrent (goroutines handle it). In Python, you must explicitly use `async/await` to avoid blocking. `httpx.AsyncClient` is the async version — there's also a sync `httpx.Client` but we don't use it in FastAPI handlers.

## Build It

### Step 1: Understand what embeddings are

An embedding is a list of numbers (a vector) that represents the *meaning* of a piece of text. The key property: **texts with similar meanings produce similar vectors.**

Let's see this in action.

In [ ]:
import httpx

OLLAMA_BASE_URL = "http://localhost:11434"

async def embed_texts(
    texts: list[str],
    ollama_base_url: str,
    model: str,
) -> list[list[float]]:
    """Embed a list of texts using Ollama's /api/embed endpoint.

    Returns a list of embedding vectors (list of floats).
    """
    if not texts:
        return []

    async with httpx.AsyncClient() as client:
        response = await client.post(
            f"{ollama_base_url}/api/embed",
            json={"model": model, "input": texts},
            timeout=120.0,
        )
        response.raise_for_status()
        data = response.json()

    return data["embeddings"]

# Embed a single sentence
vectors = await embed_texts(
    ["The quarterly revenue was 2.5 million dollars."],
    ollama_base_url=OLLAMA_BASE_URL,
    model="nomic-embed-text",
)

print(f"Number of vectors: {len(vectors)}")
print(f"Dimensions per vector: {len(vectors[0])}")
print(f"First 10 values: {vectors[0][:10]}")

### Step 2: Compute cosine similarity

Cosine similarity measures how similar two vectors are, on a scale from -1 (opposite) to 1 (identical). For embeddings, similar meaning → score close to 1.

The formula: `similarity = dot(A, B) / (|A| * |B|)`

This is the same math that Qdrant uses internally for vector search.

In [ ]:
import numpy as np

def cosine_similarity(a: list[float], b: list[float]) -> float:
    """Compute cosine similarity between two vectors."""
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Embed several sentences and compare
sentences = [
    "The quarterly revenue was 2.5 million dollars.",  # About revenue
    "The company earned 2.5M in Q3.",                   # Same meaning, different words
    "The engineering team grew to 45 people.",           # Different topic
    "What was the revenue last quarter?",                # Question about revenue
]

vectors = await embed_texts(sentences, OLLAMA_BASE_URL, "nomic-embed-text")

print("Cosine similarity matrix:\n")
print(f"{'':>50}", end="")
for i in range(len(sentences)):
    print(f"  [{i}]", end="")
print()

for i, s1 in enumerate(sentences):
    print(f"[{i}] {s1[:48]:>50}", end="")
    for j in range(len(sentences)):
        sim = cosine_similarity(vectors[i], vectors[j])
        print(f" {sim:.2f}", end="")
    print()

Look at the results:
- Sentences [0] and [1] should have **high similarity** (~0.85+) — they say the same thing in different words
- Sentences [0] and [2] should have **lower similarity** — different topics
- Sentences [0] and [3] should have **high similarity** — the question is about the same topic as the statement

This is why RAG works: embed the question, search for chunks with similar vectors, and you find relevant content regardless of exact wording.

> **Pitfall:** Claude Code sometimes hardcodes the model name (e.g., `model="nomic-embed-text"`) instead of reading it from config. In the real service, the model name comes from `settings.embedding_model` so it can be changed without editing code.

## Experiment

In [ ]:
# Experiment 1: Embed a question and find the most similar sentence
question = "How much money did the company make?"
passages = [
    "The quarterly revenue was 2.5 million dollars.",
    "The engineering team grew to 45 people.",
    "Customer satisfaction scores reached 4.8 out of 5.",
    "Infrastructure costs decreased by 12 percent.",
]

all_texts = [question] + passages
vecs = await embed_texts(all_texts, OLLAMA_BASE_URL, "nomic-embed-text")

q_vec = vecs[0]
print(f"Question: {question}\n")
for i, passage in enumerate(passages):
    sim = cosine_similarity(q_vec, vecs[i + 1])
    print(f"  {sim:.3f}  {passage}")

print(f"\nThe highest-scoring passage is the one about revenue — that's what")
print(f"Qdrant's similarity search does at scale.")

In [ ]:
# Experiment 2: What happens with empty input?
result = await embed_texts([], OLLAMA_BASE_URL, "nomic-embed-text")
print(f"Empty input returns: {result}")
print("We handle this edge case to avoid unnecessary API calls.")

## Check Your Understanding

1. **Why must both services (ingestion and chat) use the same embedding model?**

2. **What does cosine similarity measure, and why is it better than Euclidean distance for comparing text embeddings?**

3. **The `embed_texts` function uses `async with httpx.AsyncClient()`. What's the Go equivalent of this pattern, and why is it important?**